In [1]:
import pandas as pd
csv=pd.read_csv("Apprt.csv")
page_link=set(csv['link'])

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random

proxies_list = [
"6c57fc4bbe3233db7ed2__cr.np,bd,ru,ae,gb,us,af,in:6b028bc0ed2f0ca4@gw.dataimpulse.com:823",
"6c57fc4bbe3233db7ed2__cr.np,bd,ru,ae,gb,us,af,in:6b028bc0ed2f0ca4@gw.dataimpulse.com:823",
"6c57fc4bbe3233db7ed2__cr.np,bd,ru,ae,gb,us,af,in:6b028bc0ed2f0ca4@gw.dataimpulse.com:823",
"6c57fc4bbe3233db7ed2__cr.np,bd,ru,ae,gb,us,af,in:6b028bc0ed2f0ca4@gw.dataimpulse.com:823"
]

user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (X11; Linux x86_64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
]

# Static part of headers
base_headers = {
    "Accept-Encoding": "gzip, deflate"
}


def get_random_proxy():
    raw = random.choice(proxies_list)
    return {
        "http": f"http://{raw}",
        "https": f"http://{raw}"
    }

i = 1364
# end = 

# Main page
names = []
name_location = []
links = []
prices = []
description = []

# Details page

nearby_locations = []
locations_adv = []
features = []
status = []
ratings = []
ratings_list = []

visited_links = set()
skip=0

while len(names)<4000:
    url = f"https://www.99acres.com/flats-in-mumbai-ffidpage={i}"
    try:
        print(f"Fetching page {i}...")

        headers = base_headers.copy()
        headers["User-Agent"] = random.choice(user_agents)


        proxies = get_random_proxy()

        response = requests.get(url, headers=headers,proxies=proxies, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        for listing in soup.select("div.PseudoTupleRevamp__outerTupleWrap"):
            content = listing.select_one("div.PseudoTupleRevamp__contentWrapAb")
            if not content:
                continue
            
            try:
                link_tag = content.select_one("a.ellipsis")
                link = link_tag.get("href", "")
                if link and not link.startswith('http') and 'www.99acres.com' not in link:
                    link = "https://www.99acres.com" + link
            except:
                link = ''

            if not link or link in visited_links or link in page_link:
                skip+=1
                print(f"Skipping duplicate link: {skip}")
                continue

            visited_links.add(link)
            links.append(link)

            try:
                propname = content.select_one("div.PseudoTupleRevamp__headNrating a").text.strip()
            except:
                propname = ''
            names.append(propname)

            try:
                name_loc_tag = content.select_one("h2.PseudoTupleRevamp__subHeading")
                name_loc = name_loc_tag.get_text(separator=" ", strip=True)
            except:
                name_loc = ''
            name_location.append(name_loc)

            try:
                nearby_list = [elem.text.strip() for elem in content.select("span.tupleNew__unitHighlightTxt")]
            except:
                nearby_list = []
            nearby_locations.append(nearby_list)


            flat_url = link
            try:
                print(f"Fetching details page {i}...")
                dresponse = requests.get(flat_url, headers=headers,proxies=proxies, timeout=10)
                dsoup = BeautifulSoup(dresponse.content, 'html.parser')

                try:
                    facilities = []
                    for card in dsoup.select(".UniquesFacilities__xidFacilitiesCard"):
                        text_div = card.select_one("div > div")
                        facilities.append(text_div.text.strip())
                except Exception as e:
                    print(f"Facilities scraping failed: {e}")
                    facilities = []
                features.append(facilities)
                

                configurations = {}

                try:
                    for card in dsoup.select('div.configurationCards__cardContainer'):
                        try:
                            bhk = card.select_one('div.configurationCards__cardConfigBand span.list_header_semiBold').text.strip()
                        except:
                            bhk = ''
                        try:
                            building_type = card.select_one('div.configurationCards__cardConfigBand span.caption_subdued_medium').text.strip()
                        except:
                            building_type = ''
                        try:
                            area_type = card.select_one('span.configurationCards__cardAreaTypeStyle').text.strip()
                        except:
                            area_type = ''
                        try:
                            area = card.select_one('div.configurationCards__cardAreaHeading span.caption_subdued_medium').text.strip()
                        except:
                            area = ''
                        try:
                            price_range = card.select_one('div.configurationCards__cardPriceHeadingWrapper span.list_header_semiBold').text.strip()
                        except:
                            price_range = ''

                        if bhk:
                            configurations[bhk] = {
                                'building_type': building_type,
                                'area_type': area_type,
                                'area': area,
                                'price-range': price_range
                            }
                except:
                    configurations = {}
                prices.append(configurations)

                locations = []

                try:
                    for loc_card in dsoup.select('div.locAdvantagesCard__locAdCard'):
                        try:
                            name = loc_card.select_one('div.list_header_semiBold').text.strip()
                        except:
                            name = ''

                        if name:
                            locations.append(name)
                except:
                    locations = []
                locations_adv.append(locations)

                rating_div = dsoup.select_one("div.rc__ratingWrap div.display_l_semiBold")

                if rating_div:
                    rating = rating_div.text.strip()
                else:
                    rating = ''
                ratings.append(rating)

                rating_details = {}

                rating_blocks = dsoup.select(".ratingByFeature__circleWrap")
                if rating_blocks:
                    for block in rating_blocks:
                        try:
                            feature = block.select_one(".list_header_semiBold").text.strip()
                            rating_text = block.select_one(".caption_subdued_medium").text.strip()
                            rating_value = float(rating_text.split()[0])
                            rating_details[feature] = rating_value
                        except Exception as e:
                            continue
                else:
                    rating_details = {}

                ratings_list.append(rating_details)

                status_div = dsoup.find("div", class_="ConstructionStatus__phaseStatus")

                if status_div:
                    stat = status_div.text.strip()
                else:
                    stat = ''
                status.append(stat)

                if(len(names)%25==0):
                    print("Data Fetched:",len(names))


            except Exception as e:
                print(f"Failed to fetch detail page {i}: {e}")
                features.append([])
                prices.append({})
                locations_adv.append([])
                status.append('')
                ratings.append('')
                ratings_list.append({})

    except Exception as e:
        print(f"Failed to fetch page {i}: {e}")

    i += 1
    time.sleep(random.uniform(5, 10))

Fetching page 1364...
Fetching page 1365...
Skipping duplicate link: 1
Skipping duplicate link: 2
Skipping duplicate link: 3
Skipping duplicate link: 4
Skipping duplicate link: 5
Skipping duplicate link: 6
Skipping duplicate link: 7
Skipping duplicate link: 8
Skipping duplicate link: 9
Skipping duplicate link: 10
Skipping duplicate link: 11
Skipping duplicate link: 12
Skipping duplicate link: 13
Skipping duplicate link: 14
Skipping duplicate link: 15
Skipping duplicate link: 16
Skipping duplicate link: 17
Skipping duplicate link: 18
Skipping duplicate link: 19
Skipping duplicate link: 20
Skipping duplicate link: 21
Skipping duplicate link: 22
Skipping duplicate link: 23
Skipping duplicate link: 24
Fetching details page 1365...
Fetching page 1366...
Fetching page 1367...
Fetching page 1368...
Skipping duplicate link: 25
Skipping duplicate link: 26
Skipping duplicate link: 27
Skipping duplicate link: 28
Skipping duplicate link: 29
Skipping duplicate link: 30
Skipping duplicate link: 31
S

In [ ]:
import scrapy
import random
import time

class PropertySpider(scrapy.Spider):
    name = 'property_spider'
    allowed_domains = ['99acres.com']
    start_urls = [f"https://www.99acres.com/flats-in-mumbai-ffidpage={i}" for i in range(1000, 1100)]  # Adjust the range if needed

    # Proxies and User-Agent lists
    proxies_list = [
        # Add your proxy list here
    ]

    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Mozilla/5.0 (X11; Linux x86_64)",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
    ]

    # Static headers
    base_headers = {
        "Accept-Encoding": "gzip, deflate"
    }

    def get_random_proxy(self):
        raw = random.choice(self.proxies_list)
        return {
            "http": f"http://{raw}",
            "https": f"http://{raw}"
        }

    def start_requests(self):
        for url in self.start_urls:
            headers = self.base_headers.copy()
            headers["User-Agent"] = random.choice(self.user_agents)
            proxies = self.get_random_proxy()

            yield scrapy.Request(url, headers=headers, meta={'proxy': proxies}, callback=self.parse)

    def parse(self, response):
        # Scraping listings
        listings = response.css("div.PseudoTupleRevamp__outerTupleWrap")
        
        for listing in listings:
            link = listing.css("div.PseudoTupleRevamp__contentWrapAb a.ellipsis::attr(href)").get()
            if link:
                if not link.startswith('http'):
                    link = "https://www.99acres.com" + link
                
                # Passing to next callback to scrape the details page
                yield scrapy.Request(link, callback=self.parse_detail, meta={'proxy': response.meta['proxy'], 'link': link})
        
        # Check if there are more pages
        next_page = response.css("a.next::attr(href)").get()
        if next_page:
            yield scrapy.Request(next_page, headers=response.request.headers, meta={'proxy': response.meta['proxy']}, callback=self.parse)

    def parse_detail(self, response):
        # Extracting data from the details page
        link = response.meta['link']
        
        # Extracting various data fields
        name = response.css("div.PseudoTupleRevamp__headNrating a::text").get().strip()
        name_location = response.css("h2.PseudoTupleRevamp__subHeading::text").get().strip()
        nearby_locations = response.css("span.tupleNew__unitHighlightTxt::text").getall()

        features = response.css(".UniquesFacilities__xidFacilitiesCard div > div::text").getall()
        
        configurations = {}
        for card in response.css('div.configurationCards__cardContainer'):
            bhk = card.css('div.configurationCards__cardConfigBand span.list_header_semiBold::text').get()
            building_type = card.css('div.configurationCards__cardConfigBand span.caption_subdued_medium::text').get()
            area_type = card.css('span.configurationCards__cardAreaTypeStyle::text').get()
            area = card.css('div.configurationCards__cardAreaHeading span.caption_subdued_medium::text').get()
            price_range = card.css('div.configurationCards__cardPriceHeadingWrapper span.list_header_semiBold::text').get()

            if bhk:
                configurations[bhk] = {
                    'building_type': building_type,
                    'area_type': area_type,
                    'area': area,
                    'price-range': price_range
                }

        locations_adv = response.css('div.locAdvantagesCard__locAdCard div.list_header_semiBold::text').getall()
        rating = response.css("div.rc__ratingWrap div.display_l_semiBold::text").get().strip() if response.css("div.rc__ratingWrap div.display_l_semiBold::text") else ''
        
        rating_details = {}
        for block in response.css(".ratingByFeature__circleWrap"):
            feature = block.css(".list_header_semiBold::text").get()
            rating_value = block.css(".caption_subdued_medium::text").get()
            if rating_value:
                try:
                    rating_details[feature] = float(rating_value.split()[0])
                except ValueError:
                    continue
        
        status = response.css("div.ConstructionStatus__phaseStatus::text").get().strip() if response.css("div.ConstructionStatus__phaseStatus::text") else ''

        # Storing the extracted data in a dictionary
        yield {
            'name': name,
            'name_location': name_location,
            'nearby_locations': nearby_locations,
            'features': features,
            'configurations': configurations,
            'locations_adv': locations_adv,
            'rating': rating,
            'rating_details': rating_details,
            'status': status,
            'link': link
        }

        # Logging progress (Optional)
        self.logger.info(f"Successfully scraped: {name}")

    def close(self, reason):
        self.logger.info(f"Spider closed due to: {reason}")


In [8]:
import pandas as pd
data = {
    'name': names,
    'name_location': name_location,
    'link': links,
    'nearby_locations': nearby_locations,
    'features': features,
    'prices': prices,
    'locations_adv': locations_adv,
    'ratings': ratings,
    'ratings_list': ratings_list,
    'status': status
}

In [ ]:
df=pd.DataFrame(data)

In [ ]:
# final_df = pd.DataFrame(data)

In [10]:
final_df.to_csv("Apprt.csv",index=False)